In [1]:
from datasets import load_dataset

In [3]:
python_train = load_dataset("code_search_net", 'python', split='train[:1000]', cache_dir='./data')
java_train = load_dataset("code_search_net", 'java', split='train[:1000]', cache_dir='./data')
python_test = load_dataset("code_search_net", 'python', split='test[:1000]', cache_dir='./data')
java_test = load_dataset("code_search_net", 'java', split='test[:1000]', cache_dir='./data')

Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Extracting data files:   0%|          | 0/3 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/412178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/22176 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/23107 [00:00<?, ? examples/s]

Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Extracting data files:   0%|          | 0/3 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/454451 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/26909 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/15328 [00:00<?, ? examples/s]

In [13]:
import pandas as pd

from pathlib import Path

py_train_df = pd.read_csv('../data/py_train.csv', nrows=1000)
py_val_df = pd.read_csv('../data/py_val.csv', nrows=1000)
py_test_df = pd.read_csv('../data/py_test.csv', nrows=1000)

java_train_df = pd.read_csv('../data/java_train.csv', nrows=1000)
java_val_df = pd.read_csv('../data/java_val.csv', nrows=1000)
java_test_df = pd.read_csv('../data/java_test.csv', nrows=1000)


trn_df = pd.concat([py_train_df, java_train_df])
val_df = pd.concat([py_val_df, java_val_df])
tst_df = pd.concat([py_test_df, java_test_df])

# remove Unnamed: 0 column
trn_df = trn_df.drop(columns=['Unnamed: 0'])
val_df = val_df.drop(columns=['Unnamed: 0'])
tst_df = tst_df.drop(columns=['Unnamed: 0'])

tst_df.head()
trn_df.columns = [['language', 'code']]
val_df.columns = [['language', 'code']]
tst_df.columns = [['language', 'code']]
# sample = 0.1

trn_df = trn_df.sample(2000)
val_df = val_df.sample(2000)
tst_df = tst_df.sample(2000)
# len(trn_df), len(val_df), len(tst_df)

In [14]:
import util
from util import *
from transformers import AutoTokenizer

hparams = util.Hparam_Switch()

tokenizer = AutoTokenizer.from_pretrained(hparams.tokenizer_name)
tokenizer.add_tokens(['{', '}', '<java>', '<python>', '<'])
tokenizer.add_special_tokens({'mask_token': '<mask>'})

train_dataset = MethodDataset(tokenizer, trn_df, 'train')
valid_dataset = MethodDataset(tokenizer, val_df, 'valid')

# cache the dataset, so we can load it directly for training
torch.save(train_dataset, 'train_data.pt')
torch.save(valid_dataset, 'valid_data.pt')
len(train_dataset), len(valid_dataset)

  0%|          | 0/2000 [00:00<?, ?it/s]c:\Users\Risto Trajanov\anaconda3\envs\roast\lib\site-packages\transformers\tokenization_utils_base.py:2622: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(
100%|██████████| 2000/2000 [00:03<00:00, 548.44it/s]


(2000, 2000)